# 01 — Data Overview and Quality

Този notebook проверява качеството на основните данни: брой редове, последен тираж, липсващи стойности, валидност на числата и базови разпределения.


In [ ]:
from pathlib import Path
from itertools import combinations
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 80)
pd.set_option("display.width", 160)

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "data").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_DIR = PROJECT_ROOT / "data"
REPORTS_DIR = PROJECT_ROOT / "reports"
MODELS_DIR = PROJECT_ROOT / "models"


def read_csv_safe(path, **kwargs):
    path = Path(path)
    if not path.exists():
        print(f"Missing file: {path}")
        return pd.DataFrame()
    return pd.read_csv(path, **kwargs)


def number_columns(df: pd.DataFrame) -> list[str]:
    return [col for col in ["n1", "n2", "n3", "n4", "n5", "n6"] if col in df.columns]


def parse_numbers_text(value) -> list[int]:
    if pd.isna(value):
        return []
    parts = str(value).replace(";", ",").replace("|", ",").split(",")
    nums = []
    for part in parts:
        part = part.strip()
        if part.isdigit():
            nums.append(int(part))
    return nums


def extract_ticket_numbers(df: pd.DataFrame) -> pd.DataFrame:
    if df.empty:
        return df
    if len(number_columns(df)) == 6:
        return df.copy()
    text_col = next((c for c in df.columns if "numbers" in c.lower() or "числа" in c.lower()), None)
    if text_col is None:
        return df.copy()
    out = df.copy()
    parsed = out[text_col].apply(parse_numbers_text)
    for i in range(6):
        out[f"n{i+1}"] = parsed.apply(lambda nums: nums[i] if len(nums) > i else np.nan)
    return out


def show_latest_draw(df: pd.DataFrame) -> None:
    if df.empty:
        print("No draw data loaded.")
        return
    latest = df.tail(1).iloc[0]
    nums = [int(latest[col]) for col in number_columns(df)]
    print("Rows:", len(df))
    print("Latest date:", latest.get("date", "n/a"))
    print("Draw number:", latest.get("draw_number", latest.get("draw_no", "n/a")))
    print("Numbers:", nums)


def file_status(paths):
    rows = []
    for path in paths:
        path = Path(path)
        rows.append({
            "file": str(path.relative_to(PROJECT_ROOT)) if str(path).startswith(str(PROJECT_ROOT)) else str(path),
            "exists": path.exists(),
            "size_kb": round(path.stat().st_size / 1024, 2) if path.exists() else 0,
        })
    return pd.DataFrame(rows)


draws = read_csv_safe(DATA_DIR / "historical_draws.csv")
normalized = read_csv_safe(DATA_DIR / "v40_normalized_draw_events.csv")
canonical = read_csv_safe(DATA_DIR / "v41_canonical_draw_events.csv")
show_latest_draw(draws)


## Последни тиражи


In [ ]:
draws.tail(10)


## Синхрон между основните dataset файлове


In [ ]:
summary = []
for name, df in [("historical_draws", draws), ("v40_normalized", normalized), ("v41_canonical", canonical)]:
    if df.empty:
        summary.append({"dataset": name, "rows": 0, "latest_date": None, "latest_draw": None, "latest_numbers": None})
        continue
    latest = df.tail(1).iloc[0]
    nums = [int(latest[c]) for c in number_columns(df)]
    summary.append({
        "dataset": name,
        "rows": len(df),
        "latest_date": latest.get("date"),
        "latest_draw": latest.get("draw_number", latest.get("draw_no")),
        "latest_numbers": nums,
    })
pd.DataFrame(summary)


## Липсващи стойности


In [ ]:
draws.isna().sum().sort_values(ascending=False).to_frame('missing_values')


## Валидност на числата 1–49


In [ ]:
num_cols = number_columns(draws)
rows = []
for col in num_cols:
    rows.append({
        "column": col,
        "min": int(draws[col].min()),
        "max": int(draws[col].max()),
        "invalid_below_1": int((draws[col] < 1).sum()),
        "invalid_above_49": int((draws[col] > 49).sum()),
    })
pd.DataFrame(rows)


## Дублирани числа в един тираж


In [ ]:
def has_duplicate_numbers(row):
    nums = [int(row[col]) for col in num_cols]
    return len(nums) != len(set(nums))

duplicate_rows = draws[draws.apply(has_duplicate_numbers, axis=1)]
print("Rows with duplicate numbers:", len(duplicate_rows))
duplicate_rows.head()


## Честотно разпределение


In [ ]:
all_numbers = draws[num_cols].to_numpy().ravel()
freq = pd.Series(all_numbers).value_counts().sort_index()
freq_df = freq.rename_axis("number").reset_index(name="count")
display(freq_df.head())
ax = freq.plot(kind="bar", figsize=(14, 5), title="Frequency of numbers 1–49")
ax.set_xlabel("Number")
ax.set_ylabel("Count")
plt.show()


## Разпределение на сумите


In [ ]:
draw_sums = draws[num_cols].sum(axis=1)
ax = draw_sums.plot(kind="hist", bins=30, figsize=(10, 5), title="Distribution of draw sums")
ax.set_xlabel("Sum of six numbers")
plt.show()
draw_sums.describe().to_frame("sum_stats")
